# Milestone 5 — Phase 6: Calibrated XGBoost Finalist Comparison

This notebook compares two candidates using identical five-fold outer stratified CV on the original training partition only:

- **Candidate A:** tuned unweighted XGBoost, with hyperparameters re-tuned independently inside every calibration fold before generating OOF probabilities.
- **Candidate B:** the fixed Phase 4 balanced-XGBoost configuration.

Both candidates use sigmoid calibration and a training-only F1-optimal threshold. The historical 20% holdout is permanently excluded: it is not loaded into the experiment API, scored, or used for decisions. The five outer folds are descriptive validation, not a formal significance test.

## 1. Runtime setup and dataset

Run in Google Colab. Upload the original IBM Telco CSV if it is not already present. The split is reconstructed only to isolate the original training rows; the holdout return values are immediately discarded.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
GIT_BRANCH = "feature/phase-6-finalist-comparison"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

current_dir = Path.cwd()
PROJECT_ROOT = next((p for p in (current_dir, *current_dir.parents) if (p / "src" / "finalist_comparison.py").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/content/customer-churn-prediction")
    if (PROJECT_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", GIT_BRANCH], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", GIT_BRANCH], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", GIT_BRANCH], check=True)
    else:
        if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
            raise RuntimeError(f"Existing non-repository directory: {PROJECT_ROOT}")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "raw" / EXPECTED_CSV
if not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    if EXPECTED_CSV not in uploaded:
        raise FileNotFoundError(f"Upload the original file named {EXPECTED_CSV}")
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    DATA_PATH.write_bytes(uploaded[EXPECTED_CSV])

print(f"Project root: {PROJECT_ROOT}")
print(f"Training source: {DATA_PATH}")

In [ ]:
from src.preprocess import load_and_clean_data, prepare_features_target, split_data

df = load_and_clean_data(DATA_PATH)
X, y = prepare_features_target(df)
# Reconstruct the frozen split only to isolate original training rows.
# Both historical-holdout outputs are discarded immediately and never referenced.
X_train, X_holdout, y_train, y_holdout = split_data(X, y)
del X_holdout, y_holdout
del df, X, y
print(f"Training rows: {len(X_train):,}")
print(f"Training class counts: {y_train.value_counts().sort_index().to_dict()}")

## 2. Run strict nested cross-fitting

The default protocol uses five outer folds, five calibration folds within each outer-training partition, three tuning folds per randomized search, and 12 XGBoost parameter trials per search. Candidate A therefore performs nested hyperparameter searches for calibration OOF predictions and a separate search on each complete outer-training partition. This is computationally intensive; keep the protocol fixed for the primary comparison.

In [ ]:
from src.finalist_comparison import compare_calibrated_xgboost_finalists

phase6_results = compare_calibrated_xgboost_finalists(
    X_train,
    y_train,
    outer_splits=5,
    calibration_splits=5,
    tuning_splits=3,
    n_iter=12,
    random_state=42,
)

assert phase6_results["configuration"]["test_set_used"] is False
assert phase6_results["configuration"]["historical_holdout_accessed"] is False
display(phase6_results["outer_fold_results"])
display(phase6_results["summary"])
display(phase6_results["thresholds_and_parameters"][["candidate", "fold", "threshold", "training_oof_f1"]])

## 3. Export validation evidence

Upload the four CSV files for review. The paired fold deltas are descriptive only; five outer folds do not establish statistical significance. Do not evaluate or recover the historical holdout for this phase.

In [ ]:
from src.config import REPORTS_DIR

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "phase_6_outer_fold_results.csv": phase6_results["outer_fold_results"],
    "phase_6_fold_predictions.csv": phase6_results["fold_predictions"],
    "phase_6_summary.csv": phase6_results["summary"],
    "phase_6_thresholds_and_parameters.csv": phase6_results["thresholds_and_parameters"],
}
for filename, frame in outputs.items():
    frame.to_csv(REPORTS_DIR / filename, index=False)
    print(f"Saved: {REPORTS_DIR / filename}")
print("Phase 6 complete. Historical holdout was not accessed.")

## 4. Completion checklist

- [ ] Run every code cell successfully in Colab.
- [ ] Confirm five outer folds for both candidates and identical validation rows per fold.
- [ ] Confirm each candidate has exactly one outer-fold prediction for every training row.
- [ ] Review fold-level F1, paired fold deltas, ROC-AUC, Average Precision, precision, recall, accuracy, Brier score, and log loss.
- [ ] Review fold-specific thresholds and Candidate A hyperparameter stability.
- [ ] Upload all four CSV files for review.
- [ ] Record the conclusion without using the historical holdout.

**Interpretation rule:** use mean outer-fold F1 as the primary comparison, supported by paired fold differences and calibration metrics. Treat all five-fold summaries as descriptive; do not claim statistical significance.